# DB2Model — ROUTE (мультизадачный SFT) на target 7B — 3 сида

Закрывает дыру в матрице ablation: мультизадачность ROUTE мерилась только на 3B
(28.57%, один прогон), и результат попал внутрь разброса победителя — вывод об эффекте
сделать нельзя. Здесь тот же приём проверяется на **7B**, где эффект нашего метода вдвое
сильнее (+10.3 п.п. против +6.6), а уровень шума тот же — значит тест чувствительнее.

**С чем сравнивать (канон, тот же split из 91 вопроса):**

| Арм | EX |
|---|---|
| baseline 7B (схема в контексте) | 30.77% |
| sql-only 7B, 3 сида | **41.03% ± 1.37** (40.66 / 42.86 / 39.56) |
| **этот арм — multitask 7B** | ? |

Как читать результат:
- попал **внутрь 39.56–42.86** → мультизадачность по-прежнему неотличима от шума, вывод
  «отключена как выбор по умолчанию, вред не доказан» остаётся;
- **ниже 39.56 или выше 42.86 на всех трёх сидах** → эффект есть, и его знак наконец известен.

Данные: `train_multitask2695.json` — 1096 пар «вопрос → SQL» плюс 1599 вспомогательных
примеров ROUTE (548 schema linking, 547 noise correction, 504 continuation writing),
собранных `build_dataset.py --multitask` из тех же чистых пар, что у победителя.

**Перед запуском:** GPU + Internet, залить обновлённый `db2model/kaggle_input/` как Kaggle
Dataset (в нём должен быть `train_multitask2695.json`), поправить `DATA_DIR` под свой путь.
Прогон ~1.5–2 часа: датасет в 2.5 раза больше, чем у sql-only арма.


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [ ]:
import gc
import json
import re
from collections import Counter
from dataclasses import fields
from pathlib import Path

import torch

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# ПОПРАВЬ ПОД СВОЙ ДАТАСЕТ.
DATA_DIR = Path("/kaggle/input/datasets/vorange/db2model-lora")
OUT_DIR = Path("/kaggle/working")
SEEDS = [0, 1, 2]
TRAIN_FILE = "train_multitask2695.json"  # 1096 sql + 1599 вспомогательных ROUTE
TAG_PREFIX = "multitask7b"

train_pairs = json.loads((DATA_DIR / TRAIN_FILE).read_text(encoding="utf-8"))
val_pairs = json.loads((DATA_DIR / "val.json").read_text(encoding="utf-8"))
bird = json.loads((DATA_DIR / "bird_large.json").read_text(encoding="utf-8"))
DBS = ["financial", "toxicology", "codebase_community"]
profiles = {db: json.loads((DATA_DIR / f"{db}_profile.json").read_text(encoding="utf-8")) for db in DBS}
questions = [q for q in bird if q["db_id"] in DBS]


print("вопросов:", len(questions), "|", TRAIN_FILE, ":", len(train_pairs))
print("по задачам:", dict(Counter(p.get("task", "sql_generation") for p in train_pairs)))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

FENCED = re.compile(r"```(?:sql)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)
STATEMENT = re.compile(r"\b(WITH|SELECT)\b", re.IGNORECASE)


def schema_text(db):
    lines = []
    for table, info in profiles[db]["tables"].items():
        cols = ", ".join(f"{c['name']} {c['type']}" for c in info["columns"])
        lines.append(f"{table}({cols})")
    return "\n".join(lines)


def build_prompt(db, question, with_schema):
    system = f"You are a PostgreSQL expert for the database `{db}`. Return only SQL."
    if with_schema:
        system += f"\n\nSchema:\n{schema_text(db)}"
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )


def clean_sql(text):
    text = text.strip()
    fenced = FENCED.search(text)
    if fenced:
        text = fenced.group(1).strip()
    start = STATEMENT.search(text)
    if start:
        text = text[start.start() :]
    return text.strip().rstrip(";").strip()


def fresh_model():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    kw = dict(quantization_config=bnb, device_map={"": 0})
    try:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=COMPUTE_DTYPE, **kw)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=COMPUTE_DTYPE, **kw)
    m.config.use_cache = False
    return m


def predict(model, with_schema, tag):
    model.eval()
    model.config.use_cache = True
    preds = {}
    for i, q in enumerate(questions, 1):
        question = f"question: {q['question']}, evidence (may be empty): {q['evidence']}"
        inputs = tokenizer(build_prompt(q["db_id"], question, with_schema), return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        preds[str(q["question_id"])] = clean_sql(
            tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        )
        if i % 30 == 0:
            print(f"    {tag}: {i}/{len(questions)}")
    path = OUT_DIR / f"query_results_{tag}.json"
    path.write_text(json.dumps(preds, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(path)


print("считаем в", COMPUTE_DTYPE)

In [ ]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SUPPORTED = {f.name for f in fields(SFTConfig)}


def to_text(p):
    # sql_generation: схему в промпт не кладём — учим знанию о БД в весах.
    if p.get("task", "sql_generation") == "sql_generation":
        prompt = build_prompt(p["db_id"], p["question"], with_schema=False)
        return {"text": prompt + p["sql"] + tokenizer.eos_token}
    # Вспомогательные задачи ROUTE несут свои реплики: system/user/target уже
    # готовы в записи (build_multitask.py), формат тот же, что был на 3B-арме.
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": p["system"]}, {"role": "user", "content": p["user"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"text": prompt + p["target"] + tokenizer.eos_token}


def to_ds(pairs):
    return Dataset.from_list([to_text(p) for p in pairs])


def train_synth(seed):
    model = prepare_model_for_kbit_training(fresh_model())
    use_bf16 = COMPUTE_DTYPE is torch.bfloat16
    kwargs = dict(
        output_dir=str(OUT_DIR / "ckpt" / f"seed{seed}"),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=50,
        save_strategy="no",
        bf16=use_bf16,
        fp16=not use_bf16,
        optim="paged_adamw_8bit",
        dataset_text_field="text",
        report_to="none",
        max_length=512,
        seed=seed,
        data_seed=seed,
    )
    args = SFTConfig(**{k: v for k, v in kwargs.items() if k in SUPPORTED})
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=to_ds(train_pairs),
        eval_dataset=to_ds(val_pairs),
        peft_config=LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        ),
    )
    trainer.train()
    return trainer


done = {}

# baseline детерминирован (greedy, без обучения) — считаем один раз как опорную точку.
print("===== baseline =====")
base = fresh_model()
done["baseline"] = predict(base, with_schema=True, tag="baseline")
del base
gc.collect()
torch.cuda.empty_cache()

for seed in SEEDS:
    print(f"\n===== {TAG_PREFIX} seed={seed} =====")
    trainer = train_synth(seed)
    done[f"{TAG_PREFIX}_seed{seed}"] = predict(trainer.model, with_schema=False, tag=f"{TAG_PREFIX}_seed{seed}")
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

(OUT_DIR / "seed_files.json").write_text(json.dumps(done, indent=2), encoding="utf-8")
print("\nготово:", json.dumps(done, indent=2))

## Дома

Забрать `query_results_multitask7b_seed{0,1,2}.json` в `db2model/results/`, затем:

```bash
uv run python db2model/reproduce_leaderboard.py --only multitask7b
```

Скрипт сам извлечёт SQL, прогонит через единый оценщик и посчитает среднее ± σ по трём сидам.
Сравнивать с sql-only 7B: **41.03% ± 1.37**. Разница меньше 3 п.п. — шум, вывод не меняется.

Предсказания baseline ноутбук тоже пишет (`query_results_baseline.json`) — это контроль того,
что окружение не поехало: должно получиться 30.77%, как в каноне. Если не совпало — проблема
в окружении, а не в мультизадачности, и результат арма доверия не заслуживает.
